In [0]:
# Célula 1: cria o schema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
# Célula 2: mostra as colunas disponíveis em tb_info_filmes
spark.table("workspace.silver.tb_info_filmes").printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_filmes = spark.table("workspace.silver.tb_info_filmes")

# Window sem partição: só serve para gerar uma numeração sequencial única (a chave substituta)
window_spec = Window.orderBy("id_filme")

df_gold_dim_filmes = df_silver_filmes.select(
    "id_filme",
    "titulo",
    "titulo_original",
    "data_lancamento",
    "ano_lancamento",
    F.col("duracao_minutos").try_cast("int").alias("duracao_minutos"),
    "idioma_original",
    "status_filme",
    "sinopse",
    "frase_divulgacao"
).withColumn("sk_filme", F.row_number().over(window_spec))

# Reordena para deixar a chave substituta como primeira coluna
df_gold_dim_filmes = df_gold_dim_filmes.select(
    "sk_filme", "id_filme", "titulo", "titulo_original", "data_lancamento",
    "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme",
    "sinopse", "frase_divulgacao"
)

df_gold_dim_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_filmes")

print(f"Tabela gold.dim_filmes gravada com {df_gold_dim_filmes.count()} linhas.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_generos = spark.table("workspace.silver.tb_generos")

window_spec_genero = Window.orderBy("nome_genero")

df_gold_dim_generos = df_silver_generos.select("nome_genero").distinct() \
    .withColumn("sk_genero", F.row_number().over(window_spec_genero)) \
    .select("sk_genero", "nome_genero")

df_gold_dim_generos.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_generos")

print(f"Tabela gold.dim_generos gravada com {df_gold_dim_generos.count()} linhas.")

In [0]:
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_dim_generos = spark.table("workspace.gold.dim_generos")

df_gold_bridge_filme_genero = df_silver_generos \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_generos, on="nome_genero", how="inner") \
    .select("sk_filme", "sk_genero") \
    .distinct()

df_gold_bridge_filme_genero.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_genero")

print(f"Tabela gold.bridge_filme_genero gravada com {df_gold_bridge_filme_genero.count()} linhas.")

In [0]:
# Célula 1: Dimensão de Pessoas (Ator, Diretor, Roteirista)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")

df_pessoas_distintas = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .select("nome_pessoa_empresa").distinct() \
    .withColumnRenamed("nome_pessoa_empresa", "nome_pessoa")

window_spec_pessoa = Window.orderBy("nome_pessoa")

df_gold_dim_pessoas = df_pessoas_distintas \
    .withColumn("sk_pessoa", F.row_number().over(window_spec_pessoa)) \
    .select("sk_pessoa", "nome_pessoa")

df_gold_dim_pessoas.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_pessoas")

print(f"Tabela gold.dim_pessoas gravada com {df_gold_dim_pessoas.count()} linhas.")

In [0]:
# Célula 2: Dimensão de Empresas (Produtora)
df_empresas_distintas = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") == "Produtora") \
    .select("nome_pessoa_empresa").distinct() \
    .withColumnRenamed("nome_pessoa_empresa", "nome_empresa")

window_spec_empresa = Window.orderBy("nome_empresa")

df_gold_dim_empresas = df_empresas_distintas \
    .withColumn("sk_empresa", F.row_number().over(window_spec_empresa)) \
    .select("sk_empresa", "nome_empresa")

df_gold_dim_empresas.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_empresas")

print(f"Tabela gold.dim_empresas gravada com {df_gold_dim_empresas.count()} linhas.")

In [0]:
# Célula 1: Bridge Filme-Pessoa (com o papel como atributo da participação)
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_dim_pessoas = spark.table("workspace.gold.dim_pessoas")
df_dim_empresas = spark.table("workspace.gold.dim_empresas")

df_gold_bridge_filme_pessoa = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_pessoas, df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_dim_pessoas["nome_pessoa"], how="inner") \
    .select("sk_filme", "sk_pessoa", F.col("tipo_entidade").alias("tipo_participacao")) \
    .distinct()

df_gold_bridge_filme_pessoa.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_pessoa")

print(f"Tabela gold.bridge_filme_pessoa gravada com {df_gold_bridge_filme_pessoa.count()} linhas.")

In [0]:
# Célula 2: Bridge Filme-Empresa
df_gold_bridge_filme_empresa = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") == "Produtora") \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_empresas, df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_dim_empresas["nome_empresa"], how="inner") \
    .select("sk_filme", "sk_empresa") \
    .distinct()

df_gold_bridge_filme_empresa.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_empresa")

print(f"Tabela gold.bridge_filme_empresa gravada com {df_gold_bridge_filme_empresa.count()} linhas.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")

window_spec_avaliacao = Window.orderBy("sk_filme", "nome_usuario")

df_gold_fact_avaliacoes = df_silver_avaliacoes \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .select("sk_filme", "nome_usuario", "nota_usuario", "comentario_usuario") \
    .withColumn("sk_avaliacao", F.row_number().over(window_spec_avaliacao)) \
    .select("sk_avaliacao", "sk_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

df_gold_fact_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_avaliacoes")

print(f"Tabela gold.fact_avaliacoes gravada com {df_gold_fact_avaliacoes.count()} linhas.")

In [0]:
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_silver_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_silver_metricas = spark.table("workspace.silver.tb_metricas_engajamento")

df_gold_fact_desempenho = df_dim_filmes \
    .join(df_silver_financeiro, on="id_filme", how="left") \
    .join(df_silver_metricas, on="id_filme", how="left") \
    .select(
        "sk_filme",
        "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_percentual",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    )

df_gold_fact_desempenho.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_desempenho_filmes")

print(f"Tabela gold.fact_desempenho_filmes gravada com {df_gold_fact_desempenho.count()} linhas.")